# Polish benchmark: OCR engines and vision-language models

Runs seven models on the synthetic Polish dataset (TRDG, printed and handwritten fonts, *Lalka*, 22 cells x 100 images) and on
the Gold Set (all 1968 PHSF invocation scans, longest side 2048 px), then scores them. Each model runs in its own environment.

Models: Qwen2.5-VL 7B, InternVL3-8B, DeepSeek-VL2-tiny, GOT-OCR2, PaddleOCR, Tesseract, EasyOCR.

**Runtime → Change runtime type:** A100. L4 (22 GB) is enough for all seven; T4 (15 GB) skips Qwen and InternVL.

**Before running:** `research/make_bundle.sh` (the datasets are generated locally and sit in the zip), then put
`research/ocr_bundle.zip` in `MyDrive/ocr-benchmark/`. Results go to `MyDrive/ocr-benchmark/research/`. After a disconnect use
Run all again: finished models are skipped and an interrupted one continues.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!df -h /content | tail -1

from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/ocr-benchmark")
OUT = DRIVE / "research"
OUT.mkdir(parents=True, exist_ok=True)
print("results:", OUT)

In [ ]:
import shutil, zipfile

bundle = next((p for p in (DRIVE / "ocr_bundle.zip", Path("/content/ocr_bundle.zip")) if p.exists()), None)
assert bundle, "ocr_bundle.zip not found in MyDrive/ocr-benchmark or /content"
shutil.rmtree("/content/ocr", ignore_errors=True)
zipfile.ZipFile(bundle).extractall("/content/ocr")
%cd /content/ocr
!pip install -q uv jiwer editdistance scipy
!apt-get -qq install -y tesseract-ocr tesseract-ocr-pol > /dev/null
!wc -l research/data/labels.csv research/gold/data/labels.csv

In [ ]:
!python common/run_models.py --out "{OUT}" --language Polish \
    --models got paddle tesseract easyocr deepseek qwen7b internvl \
    --dataset synthetic research/data labels.csv --dataset gold research/gold/data labels.csv \
    --option max_new_tokens=512

In [ ]:
import json, pandas as pd

!python research/analyze.py --results "{OUT}"
runs = json.loads((OUT / "runs.json").read_text())
pd.DataFrame({m: {"status": r["status"], **{f"{d}_s/img": v["sec_per_image"] for d, v in r.get("datasets", {}).items()},
                  "total_min": round(r.get("total_s", 0) / 60, 1)} for m, r in runs.items()}).T